<div dir="rtl">

# ۲ – ساخت RAG ساده روی اخبار فارسی

در این نوت‌بوک، زیرمجموعه‌ی داده‌ی آماده‌شده را به چند بخش کوچک (چانک) تبدیل می‌کنید، برای هر چانک بردارهای embedding می‌سازید، یک ایندکس برداری (مثلاً با FAISS) ایجاد می‌کنید و در نهایت یک سیستم RAG ساده برای پاسخ‌گویی به سؤال‌های فارسی پیاده‌سازی می‌کنید.

</div>


In [3]:
!pip install -q faiss-cpu

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 23.7/23.7 MB 38.3 MB/s eta 0:00:00


In [4]:
# TODO: import های لازم را بنویسید
# مثال:
import pandas as pd
from pathlib import Path
import numpy as np
from sentence_transformers import SentenceTransformer
from sklearn.feature_extraction.text import TfidfVectorizer
import faiss


In [5]:
# TODO: فایل '../data/processed/news_subset.csv' را بخوانید و چند سطر اول را نمایش دهید
subset_path = Path('news_subset.csv')
df = pd.read_csv(subset_path, encoding="utf-8-sig")

print(df.shape)
df.head()

(1565, 11)


,id,title,short_link,service,subgroup,abstract,body,tags,published_datetime,agency_name,text
0,51415,گرانی به ایستگاه اتوبوس رسید/ افزایش 35 درصدی ...,http://fna.ir/ovay,جامعه,شهری,مدیرعامل شرکت واحد اتوبوس‌رانی تهران از افزایش...,محمود ترفع مدیرعامل شرکت واحد اتوبوسرانی تهران...,"محمد ترفع, مدیرعامل شرکت واحد اتوبوسرانی تهران...",2021-04-05 11:48:42,FarsNews,گرانی به ایستگاه اتوبوس رسید/ افزایش 35 درصدی ...
1,7475,بازدید بازرسان شرکت توزیع برق تهران از مراکز م...,http://fna.ir/f1xmag,اقتصادی,نفت و انرژی,شرکت توزیع نیروی برق تهران بزرگ امروز چهارشنبه...,به گزارش خبرنگار اقتصادی خبرگزاری فارس، شرکت ت...,"شرکت توزیع نیروی برق تهران بزرگ, رمزارز, برق‌ه...",2021-01-13 04:08:35,FarsNews,بازدید بازرسان شرکت توزیع برق تهران از مراکز م...
2,134333,ترتیل صفحه 212 قرآن/ پاداش بیشتر، کیفر عادلانه...,http://fna.ir/3p4b8,فرهنگ,قرآن و فعالیت های دینی,پیامبر (ص) می‌فرمایند: «اَلقُرآنُ غِنًی لاغِنی...,خبرگزاری فارس- گروه قرآن و فعالیت‌های دینی: پی...,"قرآن, پیامبر, ترتیل, سوره یونس, صفحه 212 قرآن,...",2021-09-13 12:01:00,FarsNews,ترتیل صفحه 212 قرآن/ پاداش بیشتر، کیفر عادلانه...
3,170721,آزادسازی ۹۴۰۰ مترمربع از اراضی باغی و کشاورزی ...,http://fna.ir/6ps35,استانها,سمنان,مدیر امور اراضی سازمان جهاد کشاورزی استان سمنا...,به گزارش خبرگزاری فارس از سمنان، غلامرضا خراسا...,"باغات و اراضی زراعی, جهاد کشاورزی, سازمان جهاد...",2022-02-17 01:38:37,FarsNews,آزادسازی ۹۴۰۰ مترمربع از اراضی باغی و کشاورزی ...
4,40750,آغاز مرحله پنجم رزمایش کمک‌ مؤمنانه در تهران/ ...,http://fna.ir/f33m7n,استانها,تهران,مرحله پنجم رزمایش همدلی و کمک‌های مؤمنانه توسط...,به گزارش خبرگزاری فارس از تهران، مرحله پنجم رز...,توزیع بیش از ۱۰۰ هزار بسته معیشتی و ۲۰۰ سری جه...,2021-03-10 10:51:48,FarsNews,آغاز مرحله پنجم رزمایش کمک مؤمنانه در تهران/ ...


In [6]:
# TODO: یک تابع chunking ساده بنویسید که متن (Title + Description) را به قطعه‌های مثلاً 800 کاراکتری با overlap 150 تبدیل کند
# خروجی را در یک DataFrame جدید با ستون‌های: chunk_id, doc_id, text, category, date ذخیره کنید
def chunk_text(
    text: str,
    chunk_size: int = 800,
    overlap: int = 150
):
    chunks = []
    start = 0
    text_length = len(text)

    while start < text_length:
        end = start + chunk_size
        chunk = text[start:end]
        chunks.append(chunk)
        start = end - overlap

        if start < 0:
            start = 0

    return chunks

chunk_rows = []

for doc_id, row in df.iterrows():
    category = row["service"]
    date = row["published_datetime"]
    title = str(row["title"]) if pd.notna(row["body"]) else ""
    abstract = str(row["abstract"]) if pd.notna(row["abstract"]) else ""

    text = (title + " " + abstract).strip()
    chunks = chunk_text(text)

    for i, chunk in enumerate(chunks):
        chunk_rows.append({
            "chunk_id": f"{doc_id}_{i}",
            "doc_id": doc_id,
            "text": chunk,
            "category": category,
            "date": date
        })

df_chunks = pd.DataFrame(chunk_rows)

df_chunks.head()


print("Total documents:", len(df))
print("Total chunks:", len(df_chunks))

Total documents: 1565
Total chunks: 1565


In [7]:
# TODO: دو مدل semantic embedding و lexical embedding را لود کنید و برای هر chunk دو بردار embedding بسازید
# با استفاده از یک روش دلخواه دو embedding را با یکدیکر ترکیب کنید و بردار سوم را بسازید
# همه‌ی embedding ها را در یک آرایه numpy قرار دهید
from sklearn.preprocessing import normalize
from sklearn.feature_extraction.text import TfidfVectorizer

semantic_model = SentenceTransformer("intfloat/multilingual-e5-small")

chunks_texts = df_chunks["text"].tolist()
semantic_embeddings = semantic_model.encode(chunks_texts,
                                            batch_size=32,
                                            convert_to_numpy=True,
                                            show_progress_bar=True)

semantic_embeddings = semantic_embeddings.astype("float32")

semantic_embeddings = np.array(semantic_embeddings)

tokenized_chunks = [text.split() for text in chunks_texts]

tfidf_vectorizer = TfidfVectorizer(
    max_features=384,
    ngram_range=(1, 2),
    stop_words=None
)

lexical_embeddings = tfidf_vectorizer.fit_transform(chunks_texts)
lexical_embeddings = lexical_embeddings.toarray().astype("float32")


alpha = 0.7
beta = 0.3

semantic_embeddings = normalize(semantic_embeddings, axis=1)
lexical_embeddings = normalize(lexical_embeddings, axis=1)

hybrid_embeddings = hybrid_embeddings = np.concatenate(
    [semantic_embeddings * alpha, lexical_embeddings * beta],
    axis=1
     )

hybrid_embeddings = normalize(hybrid_embeddings, axis=1).astype("float32")

print("Semantic shape:", semantic_embeddings.shape)
print("Lexical shape:", lexical_embeddings.shape)
print("Hybrid shape:", hybrid_embeddings.shape)

modules.json:   0%|          | 0.00/387 [00:00<?, ?B/s]

README.md: 0.00B [00:00, ?B/s]

sentence_bert_config.json:   0%|          | 0.00/57.0 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/655 [00:00<?, ?B/s]

model.safetensors:   0%|          | 0.00/471M [00:00<?, ?B/s]

...

tokenizer.json:   0%|          | 0.00/17.1M [00:00<?, ?B/s]

special_tokens_map.json:   0%|          | 0.00/167 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/200 [00:00<?, ?B/s]

Batches:   0%|          | 0/49 [00:00<?, ?it/s]

Semantic shape: (1565, 384)
Lexical shape: (1565, 384)
Hybrid shape: (1565, 768)


In [19]:
# TODO: با توجه به بعد embedding، یک FAISS IndexFlatIP بسازید و embedding ها را در آن اضافه کنید
semantic_embeddings_f = semantic_embeddings.astype("float32")
semantic_embeddings_f = np.ascontiguousarray(semantic_embeddings_f)
semantic_embeddings_f = normalize(semantic_embeddings_f, axis=1)

dim_sem = semantic_embeddings_f.shape[1]
index_sem = faiss.IndexFlatIP(dim_sem)
index_sem.add(semantic_embeddings_f)

print("Semantic index vectors:", index_sem.ntotal)
print("Semantic embedding dimension:", dim_sem)

lexical_embeddings_f = lexical_embeddings.astype("float32")
lexical_embeddings_f = np.ascontiguousarray(lexical_embeddings_f)
lexical_embeddings_f = normalize(lexical_embeddings_f, axis=1)

dim_lex = lexical_embeddings_f.shape[1]
index_lex = faiss.IndexFlatIP(dim_lex)
index_lex.add(lexical_embeddings_f)

print("Lexical index vectors:", index_lex.ntotal)
print("Lexical embedding dimension:", dim_lex)

alpha = 0.7
beta = 0.3
hybrid_embeddings_f = np.concatenate([semantic_embeddings_f * alpha, lexical_embeddings_f * beta], axis=1)
hybrid_embeddings_f = normalize(hybrid_embeddings_f, axis=1).astype("float32")
hybrid_embeddings_f = np.ascontiguousarray(hybrid_embeddings_f)

dim_hybrid = hybrid_embeddings_f.shape[1]
index_hybrid = faiss.IndexFlatIP(dim_hybrid)
index_hybrid.add(hybrid_embeddings_f)

print("Hybrid index vectors:", index_hybrid.ntotal)
print("Hybrid embedding dimension:", dim_hybrid)

Semantic index vectors: 1565
Semantic embedding dimension: 384
Lexical index vectors: 1565
Lexical embedding dimension: 384
Hybrid index vectors: 1565
Hybrid embedding dimension: 768


In [31]:
# TODO: تابعی به نام retrieve(question, top_k=5, embedding_model) بنویسید که:
#   1. سوال را embed کند
#   2. از FAISS نزدیک‌ترین chunk ها را پیدا کند
#   3. متن chunk های برتر را برگرداند
def retrieve(question: str,
             top_k: int = 5,
             embedding_type: str = "hybrid",  # "semantic", "lexical", "hybrid"
             semantic_model=None,
             tfidf_vectorizer=None,
             alpha=0.7,
             beta=0.3,
             index_sem=None,
             index_lex=None,
             index_hybrid=None,
             df_chunks=None):

    if embedding_type not in ["semantic", "lexical", "hybrid"]:
        raise ValueError("embedding_type باید یکی از ['semantic','lexical','hybrid'] باشد")

    if embedding_type == "semantic":
        if semantic_model is None or index_sem is None:
            raise ValueError("برای semantic، semantic_model و index_sem لازم است")
        query_vector = semantic_model.encode([question], normalize_embeddings=True)
        query_vector = np.asarray(query_vector, dtype="float32")
        query_vector = np.ascontiguousarray(query_vector)
        faiss.normalize_L2(query_vector)
        D, I = index_sem.search(query_vector, top_k)

    elif embedding_type == "lexical":
        if tfidf_vectorizer is None or index_lex is None:
            raise ValueError("برای lexical، tfidf_vectorizer و index_lex لازم است")
        query_vector = tfidf_vectorizer.transform([question]).toarray()
        query_vector = np.asarray(query_vector, dtype="float32")
        query_vector = np.ascontiguousarray(query_vector)
        faiss.normalize_L2(query_vector)
        D, I = index_lex.search(query_vector, top_k)

    else:
        if semantic_model is None or tfidf_vectorizer is None or index_hybrid is None:
            raise ValueError("برای hybrid، semantic_model، tfidf_vectorizer و index_hybrid لازم است")
        query_sem = semantic_model.encode([question], normalize_embeddings=True)
        query_tfidf = tfidf_vectorizer.transform([question]).toarray()
        query_hybrid = np.concatenate([query_sem * alpha, query_tfidf * beta], axis=1)
        query_hybrid = np.asarray(query_hybrid, dtype="float32")
        query_hybrid = np.ascontiguousarray(query_hybrid)
        faiss.normalize_L2(query_hybrid)
        query_vector = query_hybrid
        D, I = index_hybrid.search(query_vector, top_k)

    results = []
    for idx, score in zip(I[0], D[0]):
        row = df_chunks.iloc[idx]
        results.append({
            "chunk_id": row["chunk_id"],
            "text": row["text"],
            "category": row["category"],
            "date": row["date"],
            "score": float(score)
        })

    return results

In [39]:
# TODO: تابع answer(question) بنویسید که از retrieve استفاده کند
# فعلاً می‌توانید پاسخ را فقط با چسباندن متن chunk ها بسازید
def answer(question: str,
           top_k: int = 5,
           embedding_type: str = "hybrid",
           embedding_model=None,
           tfidf_vectorizer=None,
           alpha=0.7,
           beta=0.3,
           index_sem=None,
           index_lex=None,
           index_hybrid=None,
           df_chunks=None):

    top_chunks = retrieve(
        question= question,
        top_k=top_k,
        embedding_type= embedding_type,
        semantic_model=embedding_model,
        tfidf_vectorizer=tfidf_vectorizer,
        alpha=0.7,
        beta=0.3,
        index_sem=index_sem,
        index_lex=index_lex,
        index_hybrid=index_hybrid,
        df_chunks=df_chunks)

    answer_text = "\n\n".join([c["text"] for c in top_chunks])

    return answer_text

In [40]:
# TODO: چند سوال نمونه از خودتان بپرسید و ببینید خروجی معقول است یا خیر
sample_questions = [
    "چه گونه‌هایی در باغ وحش صفادشت تلف شدند؟",
    "قیمت نفت چقدر است؟",
    "فناوری‌های آینده چگونه است؟",
    "طرح شفافیت به کجا رسید؟",
    "بلیت‌فروشی جشنواره فیلم فجر از چه روزی آغاز می شود؟"
]

for q in sample_questions:
    print(f"\n--- Question: {q} ---\n")
    ans = answer(
        question=q,
        top_k=3,
        embedding_type="hybrid",
        embedding_model=semantic_model,
        tfidf_vectorizer=tfidf_vectorizer,
        alpha=0.7,
        beta=0.3,
        index_sem=index_sem,
        index_lex=index_lex,
        index_hybrid=index_hybrid,
        df_chunks=df_chunks
    )
    print(ans[:1000], "...")


--- Question: چه گونه‌هایی در باغ وحش صفادشت تلف شدند؟ ---

چه گونه‌هایی در باغ وحش صفادشت تلف شدند؟ در حالی با صدور دستور قضایی ورود بازدیدکنندگان به  باغ‌ وحش صفادشت تا اطلاع ثانوی ممنوع شده است که گونه‌های مختلفی مانند ببر، زرافه، گورخر و شیردریایی طی سال‌های اخیر در این مجموعه تلف شده‌اند.

تلف‌شدن ۱۱ رأس حیات‌وحش سالوک براثر طاعون ۱۱ رأس از حیات‌وحش سالوک اسفراین براثر بیماری طاعون نشخوارکنندگان تلف شدند.

برگزیدگان فستیوال تئاتر صحرایی تونس تجلیل شدند/ اثبات غنای تعزیه گروه تعزیه «شبیه واقعه» که مقام نخست فستیوال تئاتر صحرایی تونس را کسب کردند از سوی رئیس و مدیران حوزه هنری طی مراسمی تجلیل شدند. ...

--- Question: قیمت نفت چقدر است؟ ---

عبور قیمت نفت از 86 دلار/ افزایش 6 درصدی طی هفته گذشته از آنجایی که انتظار می رود، تقاضا برای نفت  از عرضه پیشی بگیرد و با تشدید تنش های سیاسی، قیمت نفت در بازارهای جهانی از 86 دلار بالاتر رفت.

روسیه: تلاش اوپک پلاس بر عدم نوسان شدید قیمت نفت است وزیر خارجه روسیه امروز گفت: تولیدکنندگان عضو اوپک پلاس تلاش خواهند کرد تا هیچ تغییر شدید قیمتی به و